# BlockGCN Fall Detection - Google Colab Training

**Sử dụng với Official Google Colab Extension trong VS Code**

### Requirements:
1. VS Code với Google Colab extension installed
2. Code BlockGCN đã upload lên Google Drive hoặc GitHub
3. Processed data (X_train.npy, y_train.npy, etc.)

### Runtime:
- Chọn **T4 GPU** (Free) hoặc **A100/L4** (Pro)
- Click "Select Kernel" → "Colab" → "New Colab Server"

In [ ]:
# ============================================================
# CELL 1: Setup Environment
# ============================================================
print("📦 Setting up environment...")

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Check GPU
import torch
print(f"\n✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️  WARNING: No GPU detected! Training will be very slow.")

In [ ]:
# ============================================================
# CELL 2: Upload Code (CHỌN 1 TRONG 3 OPTIONS)
# ============================================================

# ----------------------------------------------------------
# OPTION A: Clone từ GitHub (KHUYẾN NGHỊ)
# ----------------------------------------------------------
# !git clone https://github.com/YOUR_USERNAME/FALL_DETECTION.git /content/FALL_DETECTION

# ----------------------------------------------------------
# OPTION B: Copy từ Google Drive (nếu đã upload)
# ----------------------------------------------------------
!cp -r /content/drive/MyDrive/FALL_DETECTION /content/

# ----------------------------------------------------------
# OPTION C: Upload từ local (zip file)
# ----------------------------------------------------------
# !unzip /content/drive/MyDrive/BlockGCN_upload.zip -d /content/FALL_DETECTION/
# !unzip /content/drive/MyDrive/processed_data.zip -d /content/FALL_DETECTION/

# Verify
!ls -la /content/FALL_DETECTION/

In [ ]:
# ============================================================
# CELL 3: Install Dependencies
# ============================================================
print("📦 Installing BlockGCN dependencies...\n")

%cd /content/FALL_DETECTION/BlockGCN

# Install torchlight
!pip install -q -e torchlight/

# Install other dependencies
!pip install -q pyyaml einops

print("\n✅ Dependencies installed!")

# Verify imports
import sys
sys.path.append('/content/FALL_DETECTION/BlockGCN')

try:
    from model.ctrgcn import Model
    print("✅ BlockGCN model imported successfully")
except Exception as e:
    print(f"❌ Import error: {e}")

In [ ]:
# ============================================================
# CELL 4: Verify Data
# ============================================================
import numpy as np

data_dir = '/content/FALL_DETECTION/processed_data'

print("📊 Checking data files...\n")

X_train = np.load(f'{data_dir}/X_train.npy')
y_train = np.load(f'{data_dir}/y_train.npy')
X_test = np.load(f'{data_dir}/X_test.npy')
y_test = np.load(f'{data_dir}/y_test.npy')

print(f"✅ Train data: {X_train.shape} - Labels: {y_train.shape}")
print(f"✅ Test data:  {X_test.shape} - Labels: {y_test.shape}")
print(f"\nExpected shape: (N, 30, 17, 2)")
print(f"\n✅ Data verification complete!")

In [ ]:
# ============================================================
# CELL 5: Train BlockGCN - Joint Stream
# ============================================================
print("🚀 Starting BlockGCN training...\n")
print("Stream: Joint")
print("Config: config/fall-detection/default.yaml")
print("="*60)

%cd /content/FALL_DETECTION/BlockGCN

!python main.py \
    --config config/fall-detection/default.yaml \
    --phase train \
    --save-score True \
    --device 0 \
    --log-interval 10

print("\n✅ Training completed!")

In [ ]:
# ============================================================
# CELL 6 (OPTIONAL): Train All 4 Streams
# ============================================================
# Uncomment để train tất cả 4 streams cho ensemble
# Lưu ý: Mất ~2-3 giờ với T4 GPU

# %cd /content/FALL_DETECTION/BlockGCN
# !bash train_all_streams.sh

In [ ]:
# ============================================================
# CELL 7: Save Results to Google Drive
# ============================================================
from datetime import datetime
import shutil

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
save_dir = f'/content/drive/MyDrive/BlockGCN_Results/{timestamp}'

print(f"💾 Saving results to Drive...")
print(f"Target: {save_dir}\n")

# Copy work_dir results
!mkdir -p {save_dir}
!cp -r /content/FALL_DETECTION/BlockGCN/work_dir {save_dir}/

print(f"\n✅ Results saved!")
print(f"Location: {save_dir}")
print("\nFiles you can download:")
!ls -lh {save_dir}/work_dir/fall_detection/joint/

In [ ]:
# ============================================================
# CELL 8: Evaluate Model
# ============================================================
print("📊 Evaluating best checkpoint...\n")

# Tìm best checkpoint
import glob
checkpoints = glob.glob('/content/FALL_DETECTION/BlockGCN/work_dir/fall_detection/joint/*/epoch_best.pt')

if checkpoints:
    best_ckpt = checkpoints[0]
    print(f"Using checkpoint: {best_ckpt}\n")
    
    %cd /content/FALL_DETECTION/BlockGCN
    !python main.py \
        --config config/fall-detection/default.yaml \
        --phase test \
        --save-score True \
        --weights {best_ckpt} \
        --device 0
else:
    print("❌ No checkpoint found! Train the model first.")

## 📝 Notes

### Training Times (T4 GPU)
- **Joint stream**: ~30-45 phút (100 epochs)
- **All 4 streams**: ~2-3 giờ

### Colab Limits
- **Free**: 12h runtime, sau đó disconnect
- **Pro** ($9.99/month): 24h runtime, better GPUs (A100/L4)

### Tips
1. Save checkpoints thường xuyên về Drive
2. Nếu timeout, load lại checkpoint và continue training
3. Monitor GPU usage: `!nvidia-smi`
4. Check logs trong `work_dir/fall_detection/joint/`